# Order Payments - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_order_payments"
target_table = f"{catalog}.silver.olist_order_payments"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed(
        "payment_sequential",
        "payment_sequence_number"
    )
    .withColumnRenamed(
        "payment_installments",
        "payment_installment_count"
    )
    .withColumnRenamed(
        "payment_value",
        "payment_amount"
    )
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)